# 02 - Data Visualization with Matplotlib and Seaborn

This notebook introduces core visualization principles and then creates professional report charts.

A good analytical chart should:

1. Match the question being asked.
2. Use the correct chart type for the data.
3. Label axes clearly.
4. Avoid hiding outliers or important distribution patterns.
5. Save an output that can be reused in a report.

In [ ]:
# Install required libraries for a fresh notebook environment such as Google Colab.
import sys
import subprocess

for package in ["pandas", "numpy", "matplotlib", "seaborn", "sqlalchemy", "pyodbc", "python-dotenv"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"Package install skipped or failed for {package}: {exc}")

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import pyodbc
except Exception as exc:
    pyodbc = None
    print("pyodbc unavailable; SQL Server extraction will be skipped:", exc)

try:
    from dotenv import load_dotenv
    from sqlalchemy import create_engine
except Exception as exc:
    load_dotenv = None
    create_engine = None
    print("SQLAlchemy/dotenv unavailable; SQL Server extraction will be skipped:", exc)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ENV_FILE = ".env"
sns.set_theme(style="whitegrid")

In [ ]:
# Shared data loader. It tries SQL Server first, then uses generated fallback data.
def build_sample_indicator_data():
    institutions = [
        ("CBL", "Central Bank Liquidity Desk", "Maseru", "Central Bank", 920_000_000, 310_000_000, 0.345, 0.218, 0.020, 1),
        ("MCB", "Maseru Commercial Bank", "Maseru", "Commercial Bank", 610_000_000, 520_000_000, 0.278, 0.161, 0.048, 2),
        ("LMB", "Leribe Microfinance Bank", "Leribe", "Microfinance", 185_000_000, 171_000_000, 0.235, 0.139, 0.073, 3),
        ("QFB", "Quthing Finance Bank", "Quthing", "Commercial Bank", 275_000_000, 251_000_000, 0.242, 0.148, 0.067, 4),
        ("BDB", "Butha-Buthe Development Bank", "Butha-Buthe", "Development Bank", 330_000_000, 298_000_000, 0.255, 0.154, 0.061, 5),
        ("MFI", "Mafeteng Inclusion Finance", "Mafeteng", "Microfinance", 145_000_000, 139_000_000, 0.218, 0.128, 0.086, 6),
    ]
    rows = []
    observation_id = 1
    for day_number, observation_date in enumerate(pd.date_range("2026-04-01", periods=60, freq="D")):
        for code, name, region, inst_type, deposits, loans, liquidity, capital, npl, weight in institutions:
            liquidity_ratio = round(liquidity + ((day_number % 10) * 0.0021) - (0.014 if weight in (3, 6) else 0) - (0.018 if 33 <= day_number <= 42 else 0), 4)
            capital_ratio = round(capital + ((day_number % 8) * 0.0014) - (0.011 if weight == 6 else 0), 4)
            npl_ratio = round(npl + ((day_number % 9) * 0.0018) + (0.010 if 33 <= day_number <= 42 else 0), 4)
            credit_growth = round(0.0310 + ((day_number % 12) * 0.0016) - (npl * 0.0800), 4)
            rows.append({
                "ObservationID": observation_id,
                "ObservationDate": observation_date,
                "InstitutionCode": code,
                "InstitutionName": name,
                "Region": region,
                "InstitutionType": inst_type,
                "LiquidityRatio": liquidity_ratio,
                "CapitalAdequacyRatio": capital_ratio,
                "NplRatio": npl_ratio,
                "TransactionVolume": 850 + (weight * 95) + (day_number * 8) + ((day_number % 6) * 35),
                "TransactionValueLSL": (deposits * 0.028) + (day_number * 185_000) + (weight * 75_000) + ((day_number % 7) * 430_000),
                "CreditGrowthRate": credit_growth,
                "InflationRate": round(0.0470 + ((day_number % 11) * 0.0007), 4),
                "InterbankRate": round(0.0820 + ((day_number % 7) * 0.0009), 4),
                "StressFlag": int(liquidity_ratio < 0.2300 or capital_ratio < 0.1300 or npl_ratio >= 0.0850 or credit_growth < 0.0280),
            })
            observation_id += 1
    return pd.DataFrame(rows)

def load_indicator_data():
    query = """
    SELECT ObservationID, ObservationDate, InstitutionCode, InstitutionName, Region, InstitutionType,
           LiquidityRatio, CapitalAdequacyRatio, NplRatio, TransactionVolume, TransactionValueLSL,
           CreditGrowthRate, InflationRate, InterbankRate, CAST(StressFlag AS INT) AS StressFlag
    FROM m5.DailyFinancialIndicators
    ORDER BY ObservationDate, InstitutionCode;
    """
    try:
        if pyodbc is None or load_dotenv is None or create_engine is None:
            raise RuntimeError("SQL Server packages are unavailable")
        load_dotenv(BASE_DIR / ENV_FILE)
        driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
        server = os.getenv("DB_SERVER", "localhost,1433")
        database = os.getenv("DB_NAME", "TrainingDB")
        user = os.getenv("DB_USER", "sa")
        password = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
        trusted = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")
        parts = [f"DRIVER={{{driver}}}", f"SERVER={server}", f"DATABASE={database}", "Encrypt=yes", "TrustServerCertificate=yes", "Connection Timeout=5"]
        parts.append("Trusted_Connection=yes" if trusted else f"UID={user};PWD={password}")
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(';'.join(parts) + ';')}")
        data = pd.read_sql(query, engine, parse_dates=["ObservationDate"])
        source = "SQL Server"
    except Exception as exc:
        print("Using generated sample data. SQL Server unavailable:", exc)
        data = build_sample_indicator_data()
        source = "generated fallback data"
    print("Data source:", source)
    return data

df = load_indicator_data()
display(df.head())

## 1. Basic Chart Types

- Line chart: best for trends over time.
- Box plot: best for comparing distributions and outliers.
- Heatmap: best for showing correlation strength.
- Bar chart: best for comparing totals across categories.

In [ ]:
# Line chart: evaluate liquidity trend over time.
daily_liquidity = df.groupby("ObservationDate", as_index=False)["LiquidityRatio"].mean()

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=daily_liquidity, x="ObservationDate", y="LiquidityRatio", ax=ax, color="#0F766E")
ax.axhline(0.23, color="#B91C1C", linestyle="--", linewidth=1, label="Monitoring threshold")
ax.set_title("Average Liquidity Ratio Trend")
ax.set_xlabel("Observation date")
ax.set_ylabel("Liquidity ratio")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "liquidity_trend.png", dpi=160)
plt.show()

print("Chart interpretation check:")
print("This line chart is appropriate because ObservationDate is time-based and LiquidityRatio changes over time.")

In [ ]:
# Box plot: compare NPL distribution across institution types.
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="InstitutionType", y="NplRatio", hue="InstitutionType", ax=ax, palette="Set2", legend=False)
ax.set_title("NPL Ratio Distribution by Institution Type")
ax.set_xlabel("Institution type")
ax.set_ylabel("NPL ratio")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "npl_distribution_by_type.png", dpi=160)
plt.show()

print("Chart interpretation check:")
print("The box plot shows median, spread, and outliers, so it is better than a bar chart for distribution comparison.")

In [ ]:
# Heatmap: inspect correlation patterns between numeric indicators.
chart_columns = [
    "LiquidityRatio", "CapitalAdequacyRatio", "NplRatio", "CreditGrowthRate",
    "InflationRate", "InterbankRate", "TransactionValueLSL",
]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[chart_columns].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Financial Indicator Correlation Heatmap")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=160)
plt.show()

print("Chart interpretation check:")
print("The heatmap should be read by color intensity and sign. Darker colors indicate stronger relationships.")

In [ ]:
# Bar chart: compare total transaction value by region.
region_totals = (
    df.groupby("Region", as_index=False)["TransactionValueLSL"]
    .sum()
    .sort_values("TransactionValueLSL", ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=region_totals, x="TransactionValueLSL", y="Region", ax=ax, color="#2563EB")
ax.set_title("Total Transaction Value by Region")
ax.set_xlabel("Transaction value, LSL")
ax.set_ylabel("Region")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "transaction_value_by_region.png", dpi=160)
plt.show()

display(region_totals)
print("Chart interpretation check:")
print("A horizontal bar chart is readable here because region names can be scanned easily.")

## Visualisation Review Questions

Before adding a chart to a report, answer these questions:

1. Does the chart type match the data type?
2. Are labels and units clear?
3. Does the visual hide important outliers?
4. Does the title explain the chart's analytical purpose?
5. Can a reader understand the chart without reading the code?